In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# 00_setup_paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/MLP/Project5"
analysis_dir = os.path.join(PROJECT_ROOT, "analysis_outputs")
os.makedirs(analysis_dir, exist_ok=True)

lazy_exp_dir = os.path.join(
    PROJECT_ROOT,
    "final_outputs",
    "vit_b16_lazystrike_k1_expand_10ep"
)

lazy_eval_dir = os.path.join(
    lazy_exp_dir,
    "eval_jpeg_q30_controlled"
)

orig_path = os.path.join(lazy_eval_dir, "predictions_original.csv")
jpeg_path = os.path.join(lazy_eval_dir, "predictions_jpeg_q30_controlled.csv")

print("analysis_dir:", analysis_dir)
print("lazy_eval_dir:", lazy_eval_dir)
print("orig exists:", os.path.exists(orig_path), orig_path)
print("jpeg exists:", os.path.exists(jpeg_path), jpeg_path)

In [ ]:
# ============================================================
# 01_load_lazystrike_predictions
# ============================================================

orig = pd.read_csv(orig_path)
jpeg = pd.read_csv(jpeg_path)

print("orig shape:", orig.shape)
print("jpeg shape:", jpeg.shape)

print("orig columns:")
print(orig.columns.tolist())

display(orig.head())
display(jpeg.head())

In [ ]:
# ============================================================
# 02_paired_transition_analysis
# ============================================================

def add_pair_key(df):
    df = df.copy()
    df["pair_key"] = (
        df["true_label_name"].astype(str)
        + "_"
        + df["image_id"].astype(str)
    )
    return df

orig = add_pair_key(orig)
jpeg = add_pair_key(jpeg)

print("Original duplicated pair_key:", orig["pair_key"].duplicated().sum())
print("JPEG duplicated pair_key:", jpeg["pair_key"].duplicated().sum())

orig_small = orig[
    [
        "pair_key",
        "image_path",
        "filename",
        "image_id",
        "true_label_idx",
        "true_label_name",
        "pred_label_idx",
        "pred_label_name",
        "correct",
        "prob_fake",
        "prob_real",
    ]
].copy()

jpeg_small = jpeg[
    [
        "pair_key",
        "image_path",
        "filename",
        "image_id",
        "true_label_idx",
        "true_label_name",
        "pred_label_idx",
        "pred_label_name",
        "correct",
        "prob_fake",
        "prob_real",
    ]
].copy()

paired = orig_small.merge(
    jpeg_small,
    on="pair_key",
    suffixes=("_orig", "_jpeg"),
    how="inner"
)

label_mismatch = (
    paired["true_label_name_orig"] != paired["true_label_name_jpeg"]
).sum()

print("label mismatch:", label_mismatch)
print("paired shape:", paired.shape)

paired["prob_fake_drop"] = paired["prob_fake_orig"] - paired["prob_fake_jpeg"]
paired["prob_fake_change"] = paired["prob_fake_jpeg"] - paired["prob_fake_orig"]
paired["prob_real_change"] = paired["prob_real_jpeg"] - paired["prob_real_orig"]

def classify_transition(row):
    true_label = row["true_label_name_orig"]
    orig_pred = row["pred_label_name_orig"]
    jpeg_pred = row["pred_label_name_jpeg"]

    if true_label == "fake":
        if orig_pred == "fake" and jpeg_pred == "fake":
            return "fake_TP_to_TP"
        elif orig_pred == "fake" and jpeg_pred == "real":
            return "fake_TP_to_FN"
        elif orig_pred == "real" and jpeg_pred == "fake":
            return "fake_FN_to_TP"
        elif orig_pred == "real" and jpeg_pred == "real":
            return "fake_FN_to_FN"

    if true_label == "real":
        if orig_pred == "real" and jpeg_pred == "real":
            return "real_TN_to_TN"
        elif orig_pred == "real" and jpeg_pred == "fake":
            return "real_TN_to_FP"
        elif orig_pred == "fake" and jpeg_pred == "real":
            return "real_FP_to_TN"
        elif orig_pred == "fake" and jpeg_pred == "fake":
            return "real_FP_to_FP"

    return "other"

paired["transition"] = paired.apply(classify_transition, axis=1)

transition_counts = paired["transition"].value_counts().reset_index()
transition_counts.columns = ["transition", "count"]

display(transition_counts)

save_path = os.path.join(
    analysis_dir,
    "vit_b16_lazystrike_k1_paired_predictions_original_vs_jpeg.csv"
)
paired.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved paired predictions:", save_path)

In [ ]:
# ============================================================
# 03_lazystrike_transition_summary
# ============================================================

fake_subset = paired[paired["true_label_name_orig"] == "fake"]
real_subset = paired[paired["true_label_name_orig"] == "real"]

count_dict = paired["transition"].value_counts().to_dict()

lazy_transition_summary = pd.DataFrame([
    {
        "model_key": "vit_b16_lazystrike_k1",
        "model_name": "ViT-B/16 + LazyStrike-k1",
        "fake_total": len(fake_subset),
        "real_total": len(real_subset),

        "fake_TP_to_TP": count_dict.get("fake_TP_to_TP", 0),
        "fake_TP_to_FN": count_dict.get("fake_TP_to_FN", 0),
        "fake_FN_to_TP": count_dict.get("fake_FN_to_TP", 0),
        "fake_FN_to_FN": count_dict.get("fake_FN_to_FN", 0),

        "real_TN_to_TN": count_dict.get("real_TN_to_TN", 0),
        "real_TN_to_FP": count_dict.get("real_TN_to_FP", 0),
        "real_FP_to_TN": count_dict.get("real_FP_to_TN", 0),
        "real_FP_to_FP": count_dict.get("real_FP_to_FP", 0),

        "fake_TP_to_FN_ratio_percent": count_dict.get("fake_TP_to_FN", 0)
        / len(fake_subset)
        * 100,

        "mean_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].mean(),
        "median_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].median(),
        "std_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].std(),

        "mean_prob_fake_drop_real_images": real_subset["prob_fake_drop"].mean(),
        "median_prob_fake_drop_real_images": real_subset["prob_fake_drop"].median(),
        "std_prob_fake_drop_real_images": real_subset["prob_fake_drop"].std(),
    }
])

save_path = os.path.join(
    analysis_dir,
    "vit_b16_lazystrike_k1_transition_summary_original_vs_jpeg.csv"
)

lazy_transition_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
display(lazy_transition_summary)

In [ ]:
# ============================================================
# 04_compare_vit_baseline_vs_lazystrike
# ============================================================

baseline_summary_path = os.path.join(
    analysis_dir,
    "transition_summary_original_vs_jpeg.csv"
)

baseline_transition_summary = pd.read_csv(baseline_summary_path)

vit_baseline = baseline_transition_summary[
    baseline_transition_summary["model_key"] == "vit_b16"
].copy()

compare_df = pd.concat(
    [
        vit_baseline,
        lazy_transition_summary
    ],
    ignore_index=True
)

compare_cols = [
    "model_key",
    "model_name",
    "fake_total",
    "fake_TP_to_FN",
    "fake_FN_to_TP",
    "fake_TP_to_FN_ratio_percent",
    "mean_prob_fake_drop_fake_images",
    "median_prob_fake_drop_fake_images",
]

display(compare_df[compare_cols])

save_path = os.path.join(
    analysis_dir,
    "vit_baseline_vs_lazystrike_paired_comparison.csv"
)

compare_df[compare_cols].to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", save_path)

In [ ]:
# ============================================================
# Clean comparison table: ViT baseline vs LazyStrike
# ============================================================

clean_compare_df = pd.DataFrame([
    {
        "model_name": "ViT-B/16",
        "fake_total": 6000,
        "fake_TP_to_FN": 102,
        "fake_FN_to_TP": 19,
        "fake_TP_to_FN_ratio_percent": 102 / 6000 * 100,
        "mean_prob_fake_drop_fake_images": 0.016788,
        "interpretation": "Baseline"
    },
    {
        "model_name": "ViT-B/16 + LazyStrike-k1",
        "fake_total": 6000,
        "fake_TP_to_FN": 136,
        "fake_FN_to_TP": 12,
        "fake_TP_to_FN_ratio_percent": 136 / 6000 * 100,
        "mean_prob_fake_drop_fake_images": 0.019980,
        "interpretation": "Higher fake sensitivity, but larger JPEG-induced fake-to-real shift"
    },
])

save_path = os.path.join(
    analysis_dir,
    "vit_baseline_vs_lazystrike_clean_comparison.csv"
)

clean_compare_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
display(clean_compare_df)

In [ ]:
# ============================================================
# Save LazyStrike fake TP->FN candidate cases
# ============================================================

lazy_cases = paired[paired["transition"] == "fake_TP_to_FN"].copy()
lazy_cases = lazy_cases.sort_values("prob_fake_drop", ascending=False)

save_path = os.path.join(
    analysis_dir,
    "vit_b16_lazystrike_k1_fake_TP_to_FN_cases.csv"
)

lazy_cases.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)

display(
    lazy_cases[
        [
            "image_id_orig",
            "filename_orig",
            "filename_jpeg",
            "prob_fake_orig",
            "prob_fake_jpeg",
            "prob_fake_drop",
            "transition",
        ]
    ].head(10)
)